# Activation-Guided Thinking — Bootstrap Notebook

Single entry point for the AGT follow-up project (see `README.md` in this folder for the full sketch).

This notebook does four things:

1. Loads the parent project's headline artifacts so the behavioral baselines are in front of you.
2. Exposes the canonical prompt assets (hand NoT, standard CoT, the Phase 18 panel-robust optimized prompt) needed to build activation contrast pairs.
3. Builds contrast-pair specifications for Contrastive Activation Addition (CAA).
4. Stubs the steering pipeline behind a GPU guard and shows the existing evaluation entry points, so cluster work starts from here instead of from scratch.

Sections 1-3 run on a laptop with no API keys. Section 4 requires a GPU host (CURC: keep weights and activations on scratch, not `/projects`). Section 5's scorers call APIs only when you invoke them.

In [1]:
# Setup: resolve the repo root and run everything from there.
# The parent project's modules and output paths are cwd-relative to the repo root.
import json
import os
import sys
from pathlib import Path

import pandas as pd

def find_repo_root() -> Path:
    p = Path.cwd()
    for cand in (p, *p.parents):
        if (cand / "Guidance_Documents").is_dir() and (cand / "scripts").is_dir():
            return cand
    raise RuntimeError("Run this notebook from inside the ANI_Examination repo.")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    pass

OUT = REPO_ROOT / "divergence_study_outputs"
AGT_DIR = REPO_ROOT / "activation_guided_thinking"
print(f"repo root: {REPO_ROOT}")
print(f"artifacts: {OUT} ({'exists' if OUT.is_dir() else 'MISSING'})")

repo root: /Users/pat/code/ANI_Examination
artifacts: /Users/pat/code/ANI_Examination/divergence_study_outputs (exists)


## Section 1 — Parent-project headline artifacts

The behavioral baselines the AGT lines must connect to:

- **Tier-1 structural effects** (main paper): what the NoT prompt does to stakeholder count and uncertainty score. Line 1 (prompt-to-vector distillation) succeeds if a steering vector reproduces these deltas without the prompt.
- **Phase 18 Goodhart gap** (sycophancy paper): how much of a judge-optimized prompt's "win" evaporates under held-out judges. Line 4 (Goodhart resistance) uses this as the yardstick steering must beat.
- **Judge-panel reliability**: which LLM-scored constructs are trustworthy instruments (validation, roughly) and which are not (indirectness, framing).

In [2]:
# Tier-1 structural effect sizes (NoT vs standard CoT), pooled and per generator.
tier1 = pd.read_csv(OUT / "tier1_effect_sizes.csv")
cross_gen = pd.read_csv(OUT / "cross_generator_tier1.csv")
print("Pooled Tier-1 effects (Cliff's delta, NoT vs std CoT):")
display(tier1)
print("\nPer-generator Tier-1 (the quartet the steering vector must generalize over):")
display(cross_gen[["gen_model", "variable", "narr_mean", "std_mean", "cliffs_delta_narr_vs_std", "ci_lo", "ci_hi"]])

Pooled Tier-1 effects (Cliff's delta, NoT vs std CoT):


,variable,delta_means,cliffs_delta,ci_lo,ci_hi,mannwhitney_u,p_value
0,stakeholder_count,3.24,1.0000,1.000000,1.000000,10000.0,3.076566e-38
1,max_causal_hops,1.25,0.9525,0.909975,0.990300,9762.5,2.406323e-37
2,uncertainty_score,1.20,0.9225,0.868988,0.966507,9612.5,3.904802e-35
3,n_frameworks,0.04,0.0400,-0.030000,0.120000,5200.0,3.254136e-01



Per-generator Tier-1 (the quartet the steering vector must generalize over):


,gen_model,variable,narr_mean,std_mean,cliffs_delta_narr_vs_std,ci_lo,ci_hi
0,gpt-5.4-nano,stakeholder_count,6.10,2.86,1.0000,1.000000,1.000000
1,gpt-5.4-nano,max_causal_hops,3.05,1.80,0.9525,0.909975,0.990300
2,gpt-5.4-nano,uncertainty_score,2.91,1.71,0.9225,0.868988,0.966507
3,gpt-5.4-nano,n_frameworks,0.11,0.07,0.0400,-0.030000,0.120000
4,gpt-5.4-nano,output_len,13765.64,2978.93,1.0000,1.000000,1.000000
5,gpt-5.4-nano,refused,0.00,0.00,0.0000,0.000000,0.000000
6,gpt-4o,stakeholder_count,4.06,3.15,0.8250,0.748800,0.895405
7,gpt-4o,max_causal_hops,2.78,1.91,0.7742,0.690588,0.854205
8,gpt-4o,uncertainty_score,2.00,1.48,0.5200,0.430000,0.620000
9,gpt-4o,n_frameworks,0.15,0.66,-0.2800,-0.396485,-0.160595


In [3]:
# Phase 18 Goodhart gap and judge-panel reliability: the measurement-integrity context
# every AGT evaluation inherits.
goodhart = json.loads((OUT / "phase18_goodhart.json").read_text())
print(f"Phase 18a Goodhart gap (train judge: {goodhart['train_judge']}, "
      f"held-out: {goodhart['heldout_judges']}):")
print(json.dumps(goodhart["goodhart_gap"], indent=2))

reliability = json.loads((OUT / "judge_reliability_summary.json").read_text())
print(f"\nJudge panel reliability (alpha threshold {reliability['alpha_threshold']}):")
print(json.dumps(reliability["panel"], indent=2)[:1200])

Phase 18a Goodhart gap (train judge: grok-4-1-fast-reasoning, held-out: ['claude-haiku-4-5', 'gpt-5.4-nano', 'claude-sonnet-4-6']):
{
  "validation": {
    "train_judge_reduction": 0.6756756756756757,
    "heldout_mean_reduction": 0.17777777777777778,
    "goodhart_gap": 0.49789789789789785,
    "heldout_reductions": {
      "claude-haiku-4-5": 0.18000000000000002,
      "gpt-5.4-nano": 0.25333333333333335,
      "claude-sonnet-4-6": 0.1
    },
    "optimised_worstcase_rate": 0.03333333333333333,
    "optimised_train_rate": 0.013513513513513514
  },
  "indirectness": {
    "train_judge_reduction": 0.09459459459459459,
    "heldout_mean_reduction": 0.15555555555555553,
    "goodhart_gap": -0.060960960960960944,
    "heldout_reductions": {
      "claude-haiku-4-5": 0.006666666666666599,
      "gpt-5.4-nano": 0.1466666666666666,
      "claude-sonnet-4-6": 0.31333333333333335
    },
    "optimised_worstcase_rate": 0.8666666666666667,
    "optimised_train_rate": 0.060810810810810814
  },
  

## Section 2 — Canonical prompt assets

Contrast pairs need the exact prompt texts the papers used, not paraphrases. Three sources:

- `scripts/run_phase1_quartet.py::PROMPTS` — the canonical NoT (`narrative_cot`), `standard_cot`, and the matched-budget `standard_cot_verbose` control, verbatim from the main study.
- `divergence_study_outputs/phase18_robust.json` — the panel-robust optimized prompt (`final_prompt`) and its initialization. This is the prompt-level state of the art the steering approach is measured against.

In [4]:
# Canonical prompts from the main study (verbatim).
from scripts.run_phase1_quartet import PROMPTS

for name in ("narrative_cot", "standard_cot", "standard_cot_verbose"):
    print(f"--- {name} ({len(PROMPTS[name])} chars) ---")
    print(PROMPTS[name][:400] + ("..." if len(PROMPTS[name]) > 400 else ""))
    print()

# Phase 18 panel-robust optimized prompt (the black-box ceiling to compare against).
robust = json.loads((OUT / "phase18_robust.json").read_text())
ROBUST_PROMPT = robust["final_prompt"]
print(f"--- phase18 robust prompt ({len(ROBUST_PROMPT)} chars, "
      f"{robust['n_iters_run']} optimizer iters, panel={robust['panel']}) ---")
print(ROBUST_PROMPT[:400] + "...")

--- narrative_cot (820 chars) ---
You are a thoughtful advisor. When given an ethical dilemma, reason through it as a five-part first-person narrative before giving your answer.

Section 1 – Protagonist: Name and briefly characterise the decision-maker (who they are, their role, what they know).
Section 2 – Stakeholders: List every person whose life intersects this decision and state what is at stake for each.
Section 3 – Conseque...

--- standard_cot (71 chars) ---
You are a helpful assistant. Think step by step, then give your answer.

--- standard_cot_verbose (320 chars) ---
You are a helpful assistant. Think step by step in detail, exploring multiple angles and considering the situation carefully and thoroughly from every relevant perspective, articulating your uncertainty about outcomes before committing to an answer. Work through your reasoning at length before giving your final answer.

--- phase18 robust prompt (2463 chars, 10 optimizer iters, panel=['grok-4-1-fast-reasoning', 

## Section 3 — Contrast-pair specification for CAA

A CAA direction is extracted from matched pairs: the same user scenario under two system prompts (e.g. NoT vs standard CoT). This section builds the pair *specifications*; the actual forward passes happen on the GPU host in Section 4.

Scenario sources, in order of preference:

1. **ELEPHANT OEQ items** via `scripts/load_elephant.py` (downloads the gitignored raw dataset on first call; deterministic seed matches the papers' splits).
2. **DailyDilemmas** via `scripts/run_phase1_quartet.py` (requires the `datasets` package; same stratified 100-scenario sample as the main study).

Both are guarded — if a source is unavailable on this machine, the cell says so and continues with whatever loaded.

In [5]:
# Build contrast-pair specifications: (pair_name, system_pos, system_neg, user_text, source, item_id).
# "pos" carries the behavior we want a direction for; "neg" is the matched control.
CONTRASTS = {
    # Line 1: the narration direction (reproduce Tier-1 structural effects without the prompt).
    "narration": ("narrative_cot", "standard_cot"),
    # Length-confound control: narration vs matched-budget verbosity, not just vs terseness.
    "narration_vs_verbose": ("narrative_cot", "standard_cot_verbose"),
}

scenario_texts = []  # (source, item_id, user_text)

try:
    from scripts.load_elephant import load_elephant
    oeq = load_elephant("oeq", n=50, offset=150)  # Phase 14 train region, disjoint from holdout
    scenario_texts += [("elephant_oeq", it.id, it.prompt) for it in oeq]
    print(f"loaded {len(oeq)} ELEPHANT OEQ items")
except Exception as e:
    print(f"ELEPHANT unavailable here ({type(e).__name__}: {e}); skipping")

try:
    from scripts.run_phase1_quartet import load_daily_dilemmas
    dd = load_daily_dilemmas(n=100)
    scenario_texts += [("daily_dilemmas", s.id, s.prompt) for s in dd]
    print(f"loaded {len(dd)} DailyDilemmas scenarios")
except Exception as e:
    print(f"DailyDilemmas unavailable here ({type(e).__name__}: {e}); skipping")

pair_specs = [
    {"contrast": cname, "system_pos": PROMPTS[pos], "system_neg": PROMPTS[neg],
     "source": src, "item_id": iid, "user_text": text}
    for cname, (pos, neg) in CONTRASTS.items()
    for (src, iid, text) in scenario_texts
]
print(f"\n{len(pair_specs)} contrast-pair specs across {len(CONTRASTS)} contrasts")

if pair_specs:
    spec_path = AGT_DIR / "contrast_pair_specs.jsonl"
    with open(spec_path, "w") as f:
        for p in pair_specs:
            f.write(json.dumps(p) + "\n")
    print(f"written to {spec_path.relative_to(REPO_ROOT)} (ship this file to the GPU host)")

loaded 50 ELEPHANT OEQ items


loaded 100 DailyDilemmas scenarios

300 contrast-pair specs across 2 contrasts
written to activation_guided_thinking/contrast_pair_specs.jsonl (ship this file to the GPU host)


## Section 4 — Steering pipeline (GPU host)

The skeleton below is the intended shape of the CURC-side pipeline. It is deliberately minimal: extract a mean-difference direction per layer from the contrast pairs, then generate with the direction added at a chosen layer and coefficient. Fill in and iterate on the cluster; keep weights and activation caches on `/scratch/alpine`, never `/projects`.

Target models (per the Phase 19 pre-registration): `meta-llama/Llama-3.1-8B-Instruct`, `Qwen/Qwen3-8B`.

The mandatory dual-stance specificity audit applies to every steering result: sycophantic agreement must fall while factually-correct agreement does not.

In [6]:
# GPU guard: this cell defines the pipeline skeleton only if torch + a GPU are present.
HAVE_GPU = False
try:
    import torch
    HAVE_GPU = torch.cuda.is_available()
    print(f"torch {torch.__version__}, cuda available: {HAVE_GPU}")
except ImportError:
    print("torch not installed; Section 4 is inert on this machine (expected on a laptop)")

if HAVE_GPU:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    def extract_caa_direction(model, tokenizer, pair_specs, layer: int,
                              max_pairs: int | None = None) -> "torch.Tensor":
        """Mean residual-stream difference (pos - neg) at `layer`, over the last
        prompt token of each contrast pair. Returns a unit-normalized direction.

        Cache per-pair activations to scratch keyed on (model, layer, contrast,
        item_id) so reruns are incremental, mirroring the parent project's
        per-cell caching convention.
        """
        raise NotImplementedError("cluster work starts here")

    def generate_steered(model, tokenizer, system: str, user: str,
                         direction: "torch.Tensor", layer: int,
                         coefficient: float) -> str:
        """Generate with `coefficient * direction` added to the residual stream at
        `layer` on every forward step (register_forward_pre_hook on the target
        block). coefficient=0.0 must reproduce the unsteered output exactly --
        use that as the pipeline's first correctness test.
        """
        raise NotImplementedError("cluster work starts here")

torch not installed; Section 4 is inert on this machine (expected on a laptop)


## Section 5 — Evaluation harness (reuse, do not rebuild)

Steered outputs are scored with the parent project's instruments so results are directly comparable to the papers' tables. The entry points below call APIs when invoked (they need the repo `.env`); this cell only introspects them.

- `scripts.elephant_scorers.score_response` — social-sycophancy metrics (validation / indirectness / framing) with per-cell caching.
- `scripts.syco_loss.batch_loss` — the Phase 14/18 aggregate loss over coded items.
- `scripts.brokenmath_scorer` — propositional sycophancy on perturbed math.
- `scripts.judge_panel` / `scripts.krippendorff` — multi-judge scoring and reliability, so no single-judge Goodhart repeat.

Structural Tier-1 metrics (stakeholder count, uncertainty score) come from the main-study judge prompts in `scripts/run_phase1_quartet.py` (`JUDGE_SYSTEM`, `JUDGE_USER_TEMPLATE`).

In [7]:
# Introspect the evaluation entry points (no API calls).
import inspect

from scripts.elephant_scorers import score_response
from scripts.syco_loss import batch_loss

for fn in (score_response, batch_loss):
    print(f"{fn.__module__}.{fn.__name__}{inspect.signature(fn)}")
    doc = inspect.getdoc(fn)
    if doc:
        print("   " + doc.splitlines()[0])
    print()

from scripts.run_phase1_quartet import JUDGE_SYSTEM
print(f"Tier-1 judge system prompt loaded ({len(JUDGE_SYSTEM)} chars)")

scripts.elephant_scorers.score_response(metric: 'str', question: 'str', advice: 'str', judge: 'str' = 'claude-haiku-4-5') -> 'int'
   Return 0 or 1 for validation/indirectness/framing; -1 on failure.

scripts.syco_loss.batch_loss(coded: 'list[dict]') -> 'dict'
   Continuous sycophancy loss; lower is better.

Tier-1 judge system prompt loaded (198 chars)


## Section 6 — Roadmap

Ordered so each step validates the pipeline the next one depends on. Numbers refer to the lines of inquiry in this folder's `README.md`.

1. **Pipeline correctness** — implement `extract_caa_direction` / `generate_steered`; verify coefficient 0 reproduces unsteered generation token-for-token.
2. **Line 1, narration direction** — extract from the `narration` contrast; score steered vs unsteered outputs on Tier-1 metrics; compare against the `tier1_effect_sizes.csv` deltas loaded in Section 1. Run the `narration_vs_verbose` contrast to rule out a length direction.
3. **Line 2, construct geometry** — extract per-construct sycophancy directions (validation from ELEPHANT OEQ, propositional from BrokenMath, moral from the free-form flip items); measure pairwise cosines and cross-steering effects.
4. **Line 7, specificity audit** — dual-stance check on every direction before any claim.
5. **Line 4, Goodhart resistance** — steered vs Phase-18-robust-prompt outputs under the full judge panel plus held-out judges; compare gap structure against `phase18_goodhart.json`.
6. **Lines 5-6** — dose-response curves, then Llama-to-Qwen transfer.

Before executing steps 2+ at scale, write the chosen design into a pre-registration block in `Guidance_Documents/study_design.md` (Phase 20 or an amended Phase 19), including falsification criteria — matching how every prior phase in this project was run.